In [31]:
from sklearn.datasets import fetch_california_housing
import numpy as np 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

def charger_immobilier():
    data = fetch_california_housing()
    print(f"California Housing : {data.data.shape} | features : {data.feature_names} | target médiane : {np.median(data.target)*10**5:.0f} $")
    return data.data, data.target

def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    return {
        "r2":   r2_score(y_test, y_pred),
        "mae":  mean_absolute_error(y_test, y_pred),
        "rmse": root_mean_squared_error(y_test, y_pred)
    }

# --- CAS NORMAL : dataset complet ---
print("\n=== CAS NORMAL ===")
X, y = charger_immobilier()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele_lr = make_pipeline(StandardScaler(), LinearRegression())
modele_rf = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

resultats_lr = evaluer_regression(modele_lr, X_train, X_test, y_train, y_test)
resultats_rf = evaluer_regression(modele_rf, X_train, X_test, y_train, y_test)

print(f"LinearRegression : R2={resultats_lr['r2']:.2f}  MAE={resultats_lr['mae']:.2f}  RMSE={resultats_lr['rmse']:.2f}")
print(f"RandomForest     : R2={resultats_rf['r2']:.2f}  MAE={resultats_rf['mae']:.2f}  RMSE={resultats_rf['rmse']:.2f}")

# --- CAS LIMITE : 100 lignes aléatoires ---
print("\n=== CAS LIMITE : 100 lignes aléatoires ===")
np.random.seed(42)
idx = np.random.choice(len(X), 100, replace=False)
X100, y100 = X[idx], y[idx]
X_train100, X_test100, y_train100, y_test100 = train_test_split(X100, y100, test_size=0.2, random_state=42)

modele_lr2 = make_pipeline(StandardScaler(), LinearRegression())
modele_rf2 = make_pipeline(StandardScaler(), RandomForestRegressor(n_estimators=100, random_state=42))

r_lr2 = evaluer_regression(modele_lr2, X_train100, X_test100, y_train100, y_test100)
r_rf2 = evaluer_regression(modele_rf2, X_train100, X_test100, y_train100, y_test100)

print(f"LinearRegression : R2={r_lr2['r2']:.2f}  MAE={r_lr2['mae']:.2f}  RMSE={r_lr2['rmse']:.2f}")
print(f"RandomForest     : R2={r_rf2['r2']:.2f}  MAE={r_rf2['mae']:.2f}  RMSE={r_rf2['rmse']:.2f}")
print("→ R2 plus faible qu'avec le dataset complet pour le random forest : 100 lignes insuffisantes pour généraliser")
print("→ Le Random Forest mémorise les exemples d'entraînement sans généraliser")

# --- CAS ADVERSARIAL : quartier fictif ---
print("\n=== CAS ADVERSARIAL : quartier fictif ===")
# 8 variables : MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
quartier_fictif = np.array([[0, 20, 5, 1, 9000, 3, 37.0, -122.0]])

prix_lr = modele_lr.predict(quartier_fictif)[0]
prix_rf = modele_rf.predict(quartier_fictif)[0]

print(f"LinearRegression prédit : {prix_lr*100_000:.0f} $")
print(f"RandomForest     prédit : {prix_rf*100_000:.0f} $")
print("→ Valeurs potentiellement absurdes : en production il faudrait valider les entrées (revenu >= 0, population réaliste)")


=== CAS NORMAL ===
California Housing : (20640, 8) | features : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'] | target médiane : 179700 $
LinearRegression : R2=0.58  MAE=0.53  RMSE=0.75
RandomForest     : R2=0.81  MAE=0.33  RMSE=0.51

=== CAS LIMITE : 100 lignes aléatoires ===
LinearRegression : R2=0.64  MAE=0.53  RMSE=0.64
RandomForest     : R2=0.65  MAE=0.47  RMSE=0.63
→ R2 plus faible qu'avec le dataset complet pour le random forest : 100 lignes insuffisantes pour généraliser
→ Le Random Forest mémorise les exemples d'entraînement sans généraliser

=== CAS ADVERSARIAL : quartier fictif ===
LinearRegression prédit : 68895 $
RandomForest     prédit : 182437 $
→ Valeurs potentiellement absurdes : en production il faudrait valider les entrées (revenu >= 0, population réaliste)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


URL_AIRBNB = "https://data.insideairbnb.com/canada/on/ottawa/2025-09-22/visualisations/listings.csv" 


def charger_airbnb(url_csv):
    """Charge le CSV, garde les colonnes numériques utiles, nettoie les NaN.
    Doit renvoyer un DataFrame propre, sans valeurs manquantes.
    """
    df = pd.read_csv(url_csv, low_memory=False)

    colonnes = ['price', 'minimum_nights', 'number_of_reviews',
                'availability_365', 'calculated_host_listings_count']
    colonnes = [c for c in colonnes if c in df.columns]
    df = df[colonnes].copy()

    if df['price'].dtype == object:
        df['price'] = df['price'].str.replace(r'[$,]', '', regex=True).astype(float)

    avant = len(df)
    df = df.dropna()
    print(f"Listings chargés : {len(df)} lignes ({avant - len(df)} NaN supprimés), "
          f"{len(colonnes)} colonnes numériques retenues")
    return df


def choisir_k(X_scaled, k_range=range(2, 9)):
    """Pour chaque k, renvoie inertie et silhouette.
    Choisit le k au coude : plateau inertie + silhouette stable.
    """
    print(f"\n{'k':>4} | {'inertie':>10} | {'silhouette':>10}")
    print("-" * 32)
    resultats = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_)
        print(f"{k:>4} | {km.inertia_:>10.1f} | {sil:>10.3f}")
        resultats.append((k, km.inertia_, sil))

    # Choix manuel : gains d'inertie progressifs sans coude franc
    # k=5 : meilleur compromis inertie + silhouette stable (avant rechute k=7)
    # k=6 : silhouette max mais rechute à k=7 → instable
    meilleur_k = 5
    print(f"\nSegment retenu : k={meilleur_k} (choix manuel : coude inertie + silhouette stable avant rechute)")
    return meilleur_k


def decrire_clusters(df, labels):
    """Affiche les moyennes de chaque cluster et un profil en une phrase."""
    df = df.copy()
    df['cluster'] = labels
    moyennes = df.groupby('cluster').mean().round(1)
    print("\n=== DESCRIPTION DES CLUSTERS ===")
    print(moyennes.to_string())
    print("\nInterprétation :")
    for i, row in moyennes.iterrows():
        if row['price'] < moyennes['price'].median():
            profil = "petit budget"
        elif row['price'] > moyennes['price'].quantile(0.75):
            profil = "premium"
        else:
            profil = "milieu de gamme"
        print(f"  Cluster {i} → {profil} | prix moyen={row['price']:.0f} | "
              f"avis={row['number_of_reviews']:.0f} | dispo={row['availability_365']:.0f}j/an")


# CAS NORMAL
print("=" * 50)
print("CAS NORMAL")
print("=" * 50)

df = charger_airbnb(URL_AIRBNB)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

meilleur_k = choisir_k(X_scaled)

km_final = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42).fit(X_scaled)
decrire_clusters(df, km_final.labels_)


# CAS LIMITE : sans standardisation 
print("\n" + "=" * 50)
print("CAS LIMITE : KMeans SANS standardiser")
print("=" * 50)

km_raw = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42).fit(df.values)
centres_raw = pd.DataFrame(km_raw.cluster_centers_, columns=df.columns).round(1)
print("Centres des clusters (valeurs brutes) :")
print(centres_raw.to_string())
print("\n→ La colonne 'price' (grande échelle) écrase le calcul de distance :")
print("  les clusters sont presque entièrement définis par le prix,")
print("  les autres colonnes n'ont plus d'influence.")


# CAS ADVERSARIAL : valeur aberrante 
print("\n" + "=" * 50)
print("CAS ADVERSARIAL : annonce à 100 000 €/nuit")
print("=" * 50)

annonce_aberrante = pd.DataFrame(
    [[100_000, 1, 0, 365, 1]],
    columns=df.columns
)
df_pollue = pd.concat([df, annonce_aberrante], ignore_index=True)
scaler_pollue = StandardScaler()
X_pollue = scaler_pollue.fit_transform(df_pollue)

km_pollue = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42).fit(X_pollue)
centres_pollues = pd.DataFrame(
    scaler_pollue.inverse_transform(km_pollue.cluster_centers_),
    columns=df.columns
).round(1)
print("Centres après injection de l'outlier (valeurs originales) :")
print(centres_pollues.to_string())
print("\n→ Un seul point à 100 000€ tire un cluster entier vers lui et déforme tous les autres.")
print("  Le nettoyage J2 (suppression ou cap des outliers) est un prérequis absolu avant tout clustering.")

CAS NORMAL
Listings chargés : 2440 lignes (291 NaN supprimés), 5 colonnes numériques retenues

   k |    inertie | silhouette
--------------------------------
   2 |    10149.0 |      0.256
   3 |     8568.2 |      0.291
   4 |     7121.3 |      0.297
   5 |     5922.9 |      0.313
   6 |     4952.3 |      0.334
   7 |     4471.5 |      0.296
   8 |     4104.1 |      0.306

Segment retenu : k=5 (choix manuel : coude inertie + silhouette stable avant rechute)

=== DESCRIPTION DES CLUSTERS ===
          price  minimum_nights  number_of_reviews  availability_365  calculated_host_listings_count
cluster                                                                                             
0         136.7            14.3               41.3             116.5                             2.5
1         135.5            21.5               33.4             283.0                            22.1
2         144.0             7.8              335.2             217.0                             3.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# ─── FONCTIONS ───────────────────────────────────────────────────────────────

def vectoriser_textes(messages, vectorizer=None):
    if vectorizer is None:
        vectorizer = TfidfVectorizer(min_df=2)
        X = vectorizer.fit_transform(messages) #apprendre + transformer en une fois
    else:
        X = vectorizer.transform(messages) #transformer avec ce qui a deja été appris pour le test
    return X, vectorizer


def evaluer_spam(nom, modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    print(f"\n{'─'*45}")
    print(f"Modèle : {nom}")
    print(classification_report(y_test, y_pred,
                                target_names=['normal', 'spam'],
                                zero_division=0))
    return modele


# ─── HAPPY PATH ──────────────────────────────────────────────────────────────
print("=" * 50)
print("HAPPY PATH")
print("=" * 50)

messages, labels = charger_spam("SMSSpamCollection") 

msg_train, msg_test, y_train, y_test = train_test_split(
    messages, labels, test_size=0.2, random_state=42, stratify=labels
)

X_train, vec = vectoriser_textes(msg_train)
X_test,  _   = vectoriser_textes(msg_test, vectorizer=vec)

nb = evaluer_spam("Naive Bayes (MultinomialNB)",
                  MultinomialNB(),
                  X_train, X_test, y_train, y_test)

lr = evaluer_spam("Régression logistique",
                  LogisticRegression(max_iter=1000),
                  X_train, X_test, y_train, y_test)

print("\n→ Regarder le recall spam (classe minoritaire) : doit dépasser 0.85.")
print("  Un recall bas = des spams qui passent en boîte de réception sans être détectés.")


# ─── EDGE CASE : message vide ────────────────────────────────────────────────
print("\n" + "=" * 50)
print('EDGE CASE : message vide ""')
print("=" * 50)

try:
    X_vide, _ = vectoriser_textes([""], vectorizer=vec)
    proba_nb  = nb.predict_proba(X_vide)[0]
    proba_lr  = lr.predict_proba(X_vide)[0]
    pred_lr   = lr.predict(X_vide)[0]
    print(f"Naive Bayes   → proba spam={proba_nb[1]:.3f}  "
          f"décision={'spam' if proba_nb[1] > 0.5 else 'normal'}")
    print(f"Logistique    → proba spam={proba_lr[1]:.3f}  "
          f"décision={'spam' if pred_lr == 1 else 'normal'}")
    print()
    print(f"→ proba spam={proba_nb[1]:.3f} sur NB = {proba_nb[1]*100:.1f}% de chance d'être spam")
    print("→ Le vectorizer ne plante pas : message vide = vecteur tout à zéro, aucun mot connu")
    print("→ Sans mot, NB revient à la distribution de base du dataset (13% de spams)")
    print("→ Les deux modèles prédisent quand même 'normal' avec assurance")
    print("→ En production : rejeter les entrées vides en amont, le modèle ne doit pas répondre sur du vide")
except Exception as e:
    print(f"Erreur attrapée : {e}")


# ─── ADVERSARIAL : spam déguisé ──────────────────────────────────────────────
print("\n" + "=" * 50)
print("ADVERSARIAL : spam déguisé vs message normal")
print("=" * 50)

texte = "salut, ton colis t attend, confirme ici"
X_adv, _ = vectoriser_textes([texte], vectorizer=vec)

p_nb = nb.predict_proba(X_adv)[0][1]
p_lr = lr.predict_proba(X_adv)[0][1]

print(f'  "{texte}"')
print(f"  NB → proba spam={p_nb:.3f} ({'SPAM' if p_nb > 0.5 else 'normal'})")
print(f"  LR → proba spam={p_lr:.3f} ({'SPAM' if p_lr > 0.5 else 'normal'})")
print(f"\n→ proba spam={p_nb:.3f} sur NB = {p_nb*100:.1f}% de chance d'être un spam")
print("→ Le modèle se fait avoir : langage naturel sans mots-clés classiques (free, win, urgent)")
print("→ Precision : parmi les spams détectés, combien sont vrais ?")
print("→ Recall    : parmi les vrais spams, combien sont détectés ?")
print("→ Rater un vrai mail (faux positif) est souvent pire que laisser passer un spam")

HAPPY PATH
Dataset chargé : 5572 messages
  spam   = 747 (13.4%)
  normal = 4825 (86.6%)
→ Classes déséquilibrées : l'accuracy seule serait trompeuse

─────────────────────────────────────────────
Modèle : Naive Bayes (MultinomialNB)
              precision    recall  f1-score   support

      normal       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115


─────────────────────────────────────────────
Modèle : Régression logistique
              precision    recall  f1-score   support

      normal       0.97      1.00      0.99       966
        spam       1.00      0.83      0.90       149

    accuracy                           0.98      1115
   macro avg       0.99      0.91      0.95      1115
weighted avg       0.98      0.98      0.98      1115


→ Regarder le recall spam (classe

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ─── FONCTIONS ───────────────────────────────────────────────────────────────

def charger_sonar():
    """Charge le sonar, sépare X (60 colonnes) et y (M/R -> 1/0).
    Doit renvoyer X, y et afficher la répartition des classes.
    """
    dataset = fetch_ucirepo(id=151)
    X = dataset.data.features
    y = dataset.data.targets.squeeze()
    y = (y == 'M').astype(int)  # M=mine=1, R=rock=0

    mines    = y.sum()
    rochers  = len(y) - mines
    print(f"Sonar : {X.shape}, mines={mines}, rochers={rochers}")
    return X, y


def evaluer_classifieurs(X_train, X_test, y_train, y_test, standardiser=True):
    """Entraîne logistique, SVM rbf, Random Forest et affiche accuracy."""
    modeles = {
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "SVC (rbf)":          SVC(kernel="rbf"),
        "RandomForest":       RandomForestClassifier(n_estimators=100, random_state=42),
    }
    for nom, modele in modeles.items():
        if standardiser:
            pipe = make_pipeline(StandardScaler(), modele)
        else: #pour cas limite, on entraine sans scaler 
            pipe = modele
        pipe.fit(X_train, y_train)
        acc = accuracy_score(y_test, pipe.predict(X_test))
        print(f"{nom:<25} : accuracy={acc:.2f}")
    return pipe  # renvoie le dernier pour le cas adversarial


# ─── CAS NORMAL ──────────────────────────────────────────────────────────────
print("=" * 50)
print("CAS NORMAL")
print("=" * 50)

X, y = charger_sonar()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

evaluer_classifieurs(X_train, X_test, y_train, y_test, standardiser=True)


# ─── CAS LIMITE : sans standardisation ───────────────────────────────────────
print("\n" + "=" * 50)
print("CAS LIMITE : SANS standardiser")
print("=" * 50)

evaluer_classifieurs(X_train, X_test, y_train, y_test, standardiser=False)
print("→ SVM et logistique chutent sans standardisation : sensibles à l'échelle")
print("→ Random Forest moins affecté : les arbres se moquent de l'échelle")


# ─── CAS ADVERSARIAL : capteur en panne ──────────────────────────────────────
print("\n" + "=" * 50)
print("CAS ADVERSARIAL : écho à zéro (capteur en panne)")
print("=" * 50)

echo_panne = pd.DataFrame([np.zeros(60)], columns=X.columns)

modeles_entraines = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVC (rbf)":          make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True)),
    "RandomForest":       make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100, random_state=42)),
}

for nom, pipe in modeles_entraines.items():
    pipe.fit(X_train, y_train)
    pred  = pipe.predict(echo_panne)[0]
    proba = pipe.predict_proba(echo_panne)[0]
    label = "MINE" if pred == 1 else "ROCHER"
    print(f"{nom:<25} → {label} (proba mine={proba[1]:.3f})")

print("\n→ Les modèles prédisent quand même une classe avec assurance sur un signal nul")
print("→ En vrai, un sous-marin ne se fierait pas à cette prédiction")
print("→ En production : détecter en amont les échos à zéro (capteur en panne) et rejeter l'entrée")

CAS NORMAL
Sonar : (208, 60), mines=111, rochers=97
LogisticRegression        : accuracy=0.83
SVC (rbf)                 : accuracy=0.93
RandomForest              : accuracy=0.81

CAS LIMITE : SANS standardiser
LogisticRegression        : accuracy=0.81
SVC (rbf)                 : accuracy=0.83
RandomForest              : accuracy=0.81
→ SVM et logistique chutent sans standardisation : sensibles à l'échelle
→ Random Forest moins affecté : les arbres se moquent de l'échelle

CAS ADVERSARIAL : écho à zéro (capteur en panne)
LogisticRegression        → ROCHER (proba mine=0.000)
SVC (rbf)                 → ROCHER (proba mine=0.347)
RandomForest              → ROCHER (proba mine=0.280)

→ Les modèles prédisent quand même une classe avec assurance sur un signal nul
→ En vrai, un sous-marin ne se fierait pas à cette prédiction
→ En production : détecter en amont les échos à zéro (capteur en panne) et rejeter l'entrée
